# 提出動画作成用

９つのターゲットを生成し、その上にロボットの先端を移動させるタスクを実行する動画。

### arm位置の報酬巻数

##　腕の先を固定

In [1]:
# ---------------------------------------------------------
# 報酬ベースのカスタムIKリゾルバー（エルボーアップ＋鉛直固定＋姿勢の連続性）
# ---------------------------------------------------------
def get_best_ik_solution(robot, end_effector, target_pos, target_quat):
    best_q = None
    best_reward = -float('inf')
    
    # シミュレーションの現在状態を退避
    current_q_backup = robot.get_dofs_position()
    current_q_np = current_q_backup.detach().cpu().numpy().flatten() if torch.is_tensor(current_q_backup) else current_q_backup
    
    # 探索のスタート地点（シード）
    # ★追加: 現在の姿勢そのものをシードの筆頭に入れることで、連続性のある解を見つけやすくする
    seeds = [
        current_q_np, 
        np.array([0.0, -1.57, 1.57, -1.57, -1.57, 0.0]),
        np.array([1.57, -1.57, 1.57, -1.57, -1.57, 0.0]),
        np.array([-1.57, -1.57, 1.57, -1.57, -1.57, 0.0]),
        np.array([3.14, -1.57, 1.57, -1.57, -1.57, 0.0])
    ]
    
    for seed_q in seeds:
        robot.set_dofs_position(seed_q)
        ik_result = robot.inverse_kinematics(link=end_effector, pos=target_pos, quat=target_quat)
        q = ik_result.detach().cpu().numpy().flatten() if torch.is_tensor(ik_result) else ik_result
        
        # 1. 肘の高さ（エルボーアップ）評価
        elbow_reward = q[2] 
        
        # 2. 手先を鉛直（真下）に固定するペナルティ
        # q[4] (手首のピッチ) が -1.57 (-90度) からズレるほど減点
        q4_diff = np.arctan2(np.sin(q[4] - (-1.57)), np.cos(q[4] - (-1.57)))
        wrist_penalty = abs(q4_diff)
        
        # ★ 3. 姿勢の連続性ペナルティ (Minimum Joint Space Distance)
        # 移動中に手首が不必要に大回転して地面に干渉するのを防ぐため、
        # 「現在の姿勢から関節がどれくらい動くか」を計算し、変動の少ない解を優遇する
        diff = q - current_q_np
        diff = np.arctan2(np.sin(diff), np.cos(diff)) # 最短角度での移動量を計算
        movement_penalty = np.sum(np.abs(diff))
        
        # 総合スコア (手首の鉛直固定と姿勢の連続性を強く重視)
        reward = elbow_reward - (2.0 * wrist_penalty) - (1.0 * movement_penalty)
        
        if reward > best_reward:
            best_reward = reward
            # 採用する目標角度を、現在角度からの最短経路(ラップアラウンド)に補正して保存
            best_q = current_q_np + np.arctan2(np.sin(q - current_q_np), np.cos(q - current_q_np))
            
    # シミュレーション状態を元に戻す
    robot.set_dofs_position(current_q_backup)
    
    return best_q

##　旧バージョン

In [ ]:

# ---------------------------------------------------------
# 報酬ベースのカスタムIKリゾルバー（エルボーアップを能動的に選択）
# ---------------------------------------------------------
def get_best_ik_solution(robot, end_effector, target_pos, target_quat):
    best_q = None
    best_reward = -float('inf')
    
    # シミュレーションの現在状態を退避
    current_q_backup = robot.get_dofs_position()
    
    # 探索のスタート地点（シード）を4方向用意する
    seeds = [
        np.array([0.0, -1.57, 1.57, -1.57, -1.57, 0.0]),
        np.array([1.57, -1.57, 1.57, -1.57, -1.57, 0.0]),
        np.array([-1.57, -1.57, 1.57, -1.57, -1.57, 0.0]),
        np.array([3.14, -1.57, 1.57, -1.57, -1.57, 0.0])
    ]
    
    for seed_q in seeds:
        robot.set_dofs_position(seed_q)
        ik_result = robot.inverse_kinematics(link=end_effector, pos=target_pos, quat=target_quat)
        q = ik_result.detach().cpu().numpy().flatten() if torch.is_tensor(ik_result) else ik_result
        
        # --- 「報酬関数 (Reward)」の実装 ---
        # UR5eの運動学では、第3関節（q[2]: エルボージョイント）が
        # 大きなプラスの値になるほど、肘が手首より高く跳ね上がる「山折り」になります。
        # 逆にマイナスだと画像のような「谷折り」になります。
        # したがって、「q[2] の値そのものを報酬とする」ことで能動的に山折りを選ばせます。
        #　reward = q[2] 

        # --- 「報酬関数 (Reward)」の実装（拡張版） ---
        # 1. q[2] (エルボー) はプラスになるほど「山折り」で安全
        # 2. q[4] (第5関節: 手首のピッチ) は、UR5eの構造上、-1.57 (-90度) 付近だと
        #    素直に真下を向く自然な姿勢になります。プラス側に反り返る解を避けるため、
        #    -1.57からのズレが少ないほど高得点になるペナルティを引きます。
        
        elbow_reward = q[2]
        wrist_penalty = abs(q[4] - (-1.57)) 
        
        # 肘の高さを最優先しつつ、手首の反り返りペナルティを引いて総合スコアとする
        reward = elbow_reward - (0.5 * wrist_penalty)

        
        # 報酬が最も高い（一番肘が上を向いている）解を記録
        if reward > best_reward:
            best_reward = reward
            best_q = q
            
    # シミュレーション状態を元に戻す
    robot.set_dofs_position(current_q_backup)
    
    return best_q

In [2]:
import genesis as gs
import numpy as np
import imageio
import torch
import os
import random
from PIL import Image, ImageDraw

# --- 0. エラー回避用のモンキーパッチ ---
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_multiprocessor'):
    torch._C._CudaDeviceProperties.shared_memory_per_multiprocessor = 102400
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_block_optin'):
    torch._C._CudaDeviceProperties.shared_memory_per_block_optin = 99328



# ---------------------------------------------------------
# 1. ターゲット（9個）のテクスチャとXMLの動的生成
# ---------------------------------------------------------
print("1〜9の数字テクスチャとオブジェクトを動的に生成中...")
os.makedirs("textures", exist_ok=True)

# ランダムに9個の配置座標を決定（重ならないように距離を計算）
target_positions = []
while len(target_positions) < 9:
    x = random.uniform(0.35, 0.65)
    y = random.uniform(-0.25, 0.25)
    
    overlap = False
    for px, py in target_positions:
        if (px - x)**2 + (py - y)**2 < 0.08**2: # 距離0.08以内はやり直し
            overlap = True
            break
    if not overlap:
        target_positions.append((x, y))

# 各ターゲット用の画像とMujoco XML(MJCF)を生成
for i, (x, y) in enumerate(target_positions, 1):
    # ① 数字テクスチャ画像の生成 (PIL)
    img = Image.new('RGB', (64, 64), color=(240, 240, 240))
    d = ImageDraw.Draw(img)
    d.text((28, 25), str(i), fill=(30, 30, 30)) # 数字を描画
    img = img.resize((512, 512), Image.NEAREST) # ピクセルアート風に拡大
    img.save(f"textures/num_{i}.png")
    
    # ② Genesis(Mujoco)にテクスチャ付きブロックを読み込ませるためのXMLを生成
    xml_content = f"""<mujoco>
  <asset>
    <texture type="2d" name="tex_{i}" file="textures/num_{i}.png"/>
    <material name="mat_{i}" texture="tex_{i}"/>
  </asset>
  <worldbody>
    <body name="target_{i}" pos="{x} {y} 0.025">
      <geom type="box" size="0.05 0.05 0.025" material="mat_{i}"/>
    </body>
  </worldbody>
</mujoco>"""
    with open(f"target_{i}.xml", "w") as f:
        f.write(xml_content)

# ---------------------------------------------------------
# 2. 環境構築とエンティティのロード
# ---------------------------------------------------------
gs.init(backend=gs.cuda)
scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())

robot = scene.add_entity(gs.morphs.MJCF(file="mujoco_menagerie/universal_robots_ur5e/ur5e.xml"))

# さきほど自動生成したXMLから、9個のターゲットをシーンに追加
for i in range(1, 10):
    scene.add_entity(gs.morphs.MJCF(file=f"target_{i}.xml"))

cam = scene.add_camera(
    res=(640, 480),
    pos=(0.5, 0.0, 1.0),
    lookat=(0.5, 0.0, 0.0),
    fov=60,
    GUI=False
)
scene.build()
print('Genesis環境を構築し、9つのターゲットをランダム配置しました。')





# ---------------------------------------------------------
# 3. 仮想ビジョン・制御モジュール
# ---------------------------------------------------------
class PointingAI:
    def __init__(self):
        self.x_scale = 1.0 / 480
        self.y_scale = 1.0 / 640

    def project_3d_to_2d(self, X_world, Y_world):
        # 【順投影】実空間の3D座標が、カメラ画像上のどこ(u, v)に映るかを計算する
        v = 240 - (X_world - 0.5) / (self.x_scale * 0.8)
        u = 320 - (Y_world - 0.0) / (self.y_scale * 0.8)
        return int(u), int(v)

    def estimate_3d_position(self, u, v):
        # 【逆投影】カメラ画像上のピクセル座標(u, v)から、実空間のIK目標座標(X, Y, Z)を推定する
        X_world = 0.5 - (v - 240) * self.x_scale * 0.8
        Y_world = 0.0 - (u - 320) * self.y_scale * 0.8
        Z_world = 0.25 # ブロックの上空（接触回避の安全マージン）
        return np.array([X_world, Y_world, Z_world])

# ---------------------------------------------------------
# 4. メイン実行ループ
# ---------------------------------------------------------
def run_final_pointing_task():
    frames = []
    steps = 1800 # 60秒 (30fps)
    steps_per_target = 200 # 1つのターゲットにつき約6.6秒
    
    init_q = np.array([0.0, -1.57, 1.57, -1.57, -1.57, 0.0])
    robot.set_dofs_position(init_q)
    
    ai = PointingAI()
    target_quat = np.array([0.0, 1.0, 0.0, 0.0]) # 手先を下に向ける
    end_effector = robot.get_link("wrist_3_link")
    
    print('シーケンス制御による1〜9の連続ポインティングを開始します...')
    Kp = 3.0 # P制御ゲイン (少しキビキビ動かすために3.0に設定)
    
    current_target_idx = -1
    target_q = None
    
    for step in range(steps):
        # 現在どのターゲットを指差す時間帯か (0〜8)
        target_idx = min(step // steps_per_target, 8)
        
        if target_idx != current_target_idx:
            current_target_idx = target_idx
            
            # ① 配置時点の実空間座標を取得
            true_x, true_y = target_positions[current_target_idx]
            
            # ② 【仮想ビジョン】実際の座標から、画像上の「ピクセル座標 (u,v)」を擬似的に算出
            u, v = ai.project_3d_to_2d(true_x, true_y)
            
            # ③ 【仮想ビジョン】ピクセル座標 (u,v) から、IKに渡すための空間座標を推定(逆投影)
            target_pos = ai.estimate_3d_position(u, v)
            
            # ④ Genesis内蔵IKで目標の関節角度を計算
            # エルボーアップ（山折り）の姿勢ヒント
            # ④ カスタム報酬リゾルバーで、能動的に山折り姿勢を選択させる
            target_q = get_best_ik_solution(robot, end_effector, target_pos, target_quat)
                
            print(f"[{step}ステップ] ターゲット {current_target_idx + 1} (ピクセル: {u}, {v}) へ移動します。")

        # P制御 + Angle Wrapping (大回り回避) による滑らかな追従
        current_q = robot.get_dofs_position()
        if torch.is_tensor(current_q):
            current_q = current_q.detach().cpu().numpy().flatten()
            
        error = target_q[:robot.n_dofs] - current_q[:robot.n_dofs]
        error = np.arctan2(np.sin(error), np.cos(error)) # 角度の最短経路補正
        
        velocity = Kp * error
        robot.control_dofs_velocity(velocity)
        scene.step()
        
        # 録画処理
        rgb_frame, _, _, _ = cam.render()
        if torch.is_tensor(rgb_frame):
            rgb_frame = rgb_frame.detach().cpu().numpy()
        frames.append(rgb_frame)
        
    print('動画のレンダリング中...')
    video_filename = 'pai_final_task_1to9.mp4'
    imageio.mimsave(video_filename, frames, fps=30)
    print(f'動画を保存しました: {video_filename}')

# 実行
run_final_pointing_task()

e:\matuo-PYZERO\.venv\Lib\site-packages\genesis\__init__.py:41: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


1〜9の数字テクスチャとオブジェクトを動的に生成中...
[Genesis] [08:56:55] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[Genesis] [08:56:55] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [08:56:55] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [08:56:55] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [08:56:55] [INFO] Running on [NVIDIA GeForce RTX 3070 Ti] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [08:56:55] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [08:56:55] [INFO] Scene <05b4829> created.
[Genesis] [08:56:55] [INFO] Adding <gs.engine.entities.RigidEntity> plane_aa0c1a1. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [08:56:56] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properti

e:\matuo-PYZERO\.venv\Lib\site-packages\quadrants\lang\ast\ast_transformers\function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [08:57:59] [INFO] Building visualizer...
Genesis環境を構築し、9つのターゲットをランダム配置しました。
シーケンス制御による1〜9の連続ポインティングを開始します...
[0ステップ] ターゲット 1 (ピクセル: 294, 200) へ移動します。
[Genesis] [08:58:04] [INFO] Running at 0.42 FPS.
[Genesis] [08:58:04] [INFO] Running at 2.56 FPS.
[Genesis] [08:58:04] [INFO] Running at 4.71 FPS.
[Genesis] [08:58:04] [INFO] Running at 7.27 FPS.
[Genesis] [08:58:04] [INFO] Running at 8.89 FPS.
[Genesis] [08:58:04] [INFO] Running at 12.33 FPS.
[Genesis] [08:58:04] [INFO] Running at 16.38 FPS.
[Genesis] [08:58:04] [INFO] Running at 20.15 FPS.
[Genesis] [08:58:04] [INFO] Running at 23.51 FPS.
[Genesis] [08:58:04] [INFO] Running at 26.87 FPS.
[Genesis] [08:58:04] [INFO] Running at 30.21 FPS.
[Genesis] [08:58:05] [INFO] Running at 33.25 FPS.
[Genesis] [08:58:05] [INFO] Running at 36.27 FPS.
[Genesis] [08:58:05] [INFO] Running at 38.88 FPS.
[Genesis] [08:58:05] [INFO] Running at 41.42 FPS.
[Genesis] [08:58:05] [INFO] Running at 43.10 FPS.
[Genesis] [08:58:05] [INFO] Running at 45.27 